# Reconhecedor de Gatos e Cachorros

Notebook simples para treinar uma CNN (rede neural convolucional) que classifica imagens como **gato** ou **cachorro**.

Feito para rodar no Google Colab. Rode as células em ordem.

Fluxo:
1. Baixar o dataset público (microsoft/cats_vs_dogs, via Hugging Face `datasets`)
2. Explorar rapidamente os dados
3. Montar um pipeline de pré-processamento (resize + normalização + augmentation)
4. Treinar uma CNN simples
5. Avaliar (accuracy, matriz de confusão) no conjunto de teste
6. Fazer upload de uma imagem sua e classificar (gato ou cachorro)

> Dica: no Colab, ative GPU em `Ambiente de execução > Alterar tipo de ambiente de execução > GPU` para treinar mais rápido.

## 1. Imports

In [ ]:
!pip install -q -U datasets

In [ ]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from datasets import load_dataset

print("TensorFlow version:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))

## 2. Download do dataset

As URLs antigas do Google (`mledu-datasets`) e o dataset `cats_vs_dogs` do `tensorflow_datasets` apresentaram instabilidade. Por isso usamos o dataset **microsoft/cats_vs_dogs** hospedado no Hugging Face Hub, via biblioteca `datasets` — download direto, sem autenticação, com as ~1.700 imagens corrompidas do dataset original já removidas. São 23.410 imagens de gatos e cachorros.

In [ ]:
full_ds = load_dataset("microsoft/cats_vs_dogs", split="train")

class_names = full_ds.features["labels"].names  # ['cat', 'dog']
print("Classes:", class_names)
print("Total de imagens no dataset:", len(full_ds))

# Divide em treino (80%) / validação (10%) / teste (10%), mantendo a proporção de classes
split_1 = full_ds.train_test_split(test_size=0.2, seed=42, stratify_by_column="labels")
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="labels")

train_hf = split_1["train"]
val_hf = split_2["train"]
test_hf = split_2["test"]

print("Treino:", len(train_hf), "| Validação:", len(val_hf), "| Teste:", len(test_hf))

## 3. Exploração rápida dos dados

In [ ]:
for nome, d in [("treino", train_hf), ("validação", val_hf), ("teste", test_hf)]:
    print(f"{nome}: {len(d)} imagens")

In [ ]:
# Visualiza algumas imagens de exemplo
plt.figure(figsize=(10, 10))
for i in range(9):
    exemplo = train_hf[i]
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(exemplo["image"])
    plt.title(class_names[exemplo["labels"]])
    plt.axis("off")
plt.show()

## 4. Pipeline de pré-processamento

Convertemos cada imagem PIL para RGB e redimensionamos para 160x160 (as imagens do dataset têm tamanhos variados). Guardamos como array `uint8` em memória (rápido de reusar a cada época) e, no `tf.data.Dataset`, normalizamos os pixels para [0,1] e aplicamos data augmentation leve (flip horizontal + rotação) apenas no treino.

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

def para_numpy(hf_dataset):
    n = len(hf_dataset)
    imagens = np.zeros((n, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.uint8)
    labels = np.zeros((n,), dtype=np.float32)
    for i, exemplo in enumerate(hf_dataset):
        img = exemplo["image"].convert("RGB").resize(IMG_SIZE)
        imagens[i] = np.array(img)
        labels[i] = exemplo["labels"]
    return imagens, labels

print("Convertendo imagens (pode levar alguns minutos)...")
train_imgs, train_labels = para_numpy(train_hf)
val_imgs, val_labels = para_numpy(val_hf)
test_imgs, test_labels = para_numpy(test_hf)
print("Concluído.")

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

def normaliza(image, label, augment=False):
    image = tf.cast(image, tf.float32) / 255.0
    if augment:
        image = data_augmentation(image)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_imgs, train_labels))
    .shuffle(1000)
    .map(lambda x, y: normaliza(x, y, augment=True), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_imgs, val_labels))
    .map(lambda x, y: normaliza(x, y, augment=False), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_imgs, test_labels))
    .map(lambda x, y: normaliza(x, y, augment=False), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

## 5. Modelo (CNN simples)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),  # 0 = gato, 1 = cachorro
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 6. Treinamento

In [ ]:
EPOCHS = 15

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop],
)

## 7. Avaliação

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="treino")
plt.plot(epochs_range, val_acc, label="validação")
plt.legend()
plt.title("Acurácia")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="treino")
plt.plot(epochs_range, val_loss, label="validação")
plt.legend()
plt.title("Loss")
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten().tolist())
    y_pred.extend((preds.flatten() > 0.5).astype(int).tolist())

print(classification_report(y_true, y_pred, target_names=class_names))
print("Matriz de confusão:")
print(confusion_matrix(y_true, y_pred))

## 8. Testando com sua própria imagem

Rode a célula abaixo, clique em "Escolher arquivos" e envie uma foto de gato ou cachorro.

In [ ]:
from google.colab import files

def prever_imagem(caminho):
    img = tf.keras.utils.load_img(caminho, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)
    pred = model.predict(arr, verbose=0)[0][0]
    classe = class_names[1] if pred > 0.5 else class_names[0]
    confianca = pred if pred > 0.5 else 1 - pred
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Predição: {classe} ({confianca:.1%} de confiança)")
    plt.show()
    return classe, float(confianca)

uploaded = files.upload()
for nome_arquivo in uploaded.keys():
    prever_imagem(nome_arquivo)

## 9. Salvando o modelo

Salva o modelo treinado para poder reutilizar depois sem re-treinar.

In [ ]:
model.save("cats_vs_dogs_model.keras")
print("Modelo salvo em cats_vs_dogs_model.keras")